# Week 16: Distributed Training & Model Optimization
## Notebook 07: Pipeline Parallelism: 1F1B Scheduling & Interleaved Stages

### 1. Conceptual Intuition & Theoretical Background
When training models that exceed single-GPU memory capacity, **Pipeline Parallelism (PP)** partitions model layers across $p$ pipeline stages (GPUs).

**1. Naive GPipe Schedule:**
Pushes all $m$ micro-batches forward, then all backward.
- **Bubble Fraction:**
$$F_{\text{GPipe}} = \frac{p - 1}{m + p - 1}$$
- Requires storing activations for all $m$ micro-batches simultaneously, leading to severe activation memory pressure.

**2. 1F1B Schedule (One-Forward-One-Backward):**
Once the pipeline reaches steady state, each GPU alternates between executing **one forward** and **one backward** step:
- Activations are discarded as soon as backward completes, capping active activations to at most $p$ micro-batches.

**3. Interleaved 1F1B (Megatron-LM):**
Each device manages $v$ virtual stages (e.g. GPU 0 handles layers 1-2 and layers 9-10). Bubble fraction drops to:
$$F_{\text{interleaved}} \approx \frac{p - 1}{v \cdot m}$$

---

### 2. Scratch Python Implementation: 1F1B Gantt Schedule Simulator

In [ ]:
import numpy as np

def simulate_pipeline_bubble(num_stages=4, num_microbatches=16, interleaved_virtual_stages=1):
    """
    Computes pipeline bubble fraction and maximum active activations in memory.
    """
    p = num_stages
    m = num_microbatches
    v = interleaved_virtual_stages
    
    # GPipe Bubble Fraction
    bubble_gpipe = (p - 1) / (m + p - 1)
    
    # 1F1B Bubble Fraction
    bubble_1f1b = (p - 1) / (m + p - 1)
    
    # Interleaved 1F1B Bubble Fraction
    bubble_interleaved = (p - 1) / (v * m + p - 1)
    
    # Peak Activation Memory (in micro-batch units)
    mem_gpipe = m
    mem_1f1b = p
    
    return {
        'GPipe Bubble %': bubble_gpipe * 100,
        '1F1B Bubble %': bubble_1f1b * 100,
        'Interleaved 1F1B Bubble %': bubble_interleaved * 100,
        'GPipe Peak Memory': mem_gpipe,
        '1F1B Peak Memory': mem_1f1b
    }

results = simulate_pipeline_bubble(num_stages=8, num_microbatches=32, interleaved_virtual_stages=2)

print("--- Pipeline Parallelism Schedule Comparison ---")
print(f"Pipeline Stages (GPUs): 8 | Microbatches: 32")
print(f"GPipe Bubble Overhead:             {results['GPipe Bubble %']:.2f}% | Max Activation Memory: {results['GPipe Peak Memory']} chunks")
print(f"1F1B Steady-State Bubble Overhead: {results['1F1B Bubble %']:.2f}% | Max Activation Memory: {results['1F1B Peak Memory']} chunks")
print(f"Interleaved 1F1B (v=2) Overhead:   {results['Interleaved 1F1B Bubble %']:.2f}% | Max Activation Memory: {results['1F1B Peak Memory']} chunks")
print("Interleaved 1F1B halves the pipeline bubble while keeping activation memory strictly bounded!")

### 3. Step-by-Step Code Walkthrough
1. **Steady-State Balance**: In 1F1B, completing a backward pass immediately frees its forward activations.
2. **Memory Ceiling**: Memory usage is capped at $p$ instead of unbounded $m$.
3. **Interleaved Virtual Stages**: Breaks layers into smaller chunks, cutting idle bubble overhead by $v\times$.

---

### 4. Industry Applications & Trade-offs
- **LLaMA & Megatron-LM Pretraining:** Used in multi-node clusters combining PP, TP (Tensor Parallel), and DP.
- **Trade-off:** Interleaved stages increase peer-to-peer point-to-point network communication volume.

---

### 5. Top Interview Questions & Answers
**Q1: Why does the 1F1B schedule save substantial memory compared to GPipe?**
> **Answer:** GPipe executes all forward passes before starting backward passes, forcing the GPU to buffer all $m$ activation states. 1F1B pairs each forward step with a backward step, bounding peak memory to only $p$ active micro-batches.

**Q2: What is the primary downside of using Interleaved 1F1B?**
> **Answer:** Devices must communicate across virtual stages more frequently, increasing peer-to-peer communication bandwidth pressure across PCIe or NVLink.